# parse_instruction_ver20: 動作確認ノートブック

## 目的
`src/parse/instruction_parser_ver19.py` と `src/submit_baseline_ver05.py` の動作を検証する。

1. instruction 読み込み・タスク分解の確認
2. subject 混入率ゼロの確認
3. 動画処理パイプラインの動作確認（3本サンプル）
4. 処理後フレーム数・解像度の一致確認
5. ログ出力確認

In [ ]:

# Section 1: セットアップ
import sys
import json
import logging
from pathlib import Path

WORKSPACE = Path("/workspace")
NOTEBOOK_DIR = Path(__file__) if "__file__" in dir() else WORKSPACE / "notebook"
sys.path.insert(0, str(WORKSPACE))

# パス定義
ANNOTATIONS_PATH = WORKSPACE / "data" / "annotations.jsonl"
GT_PATH = WORKSPACE / "data" / "annotations_gt_task_ver10.json"
VIDEO_DIR = WORKSPACE / "data" / "videos"
TASK_RULES_PATH = WORKSPACE / "configs" / "task_rules_ver05.json"
OUTPUT_DIR = WORKSPACE / "logs" / "submit" / "submission_ver05_preview"
LOG_DIR = WORKSPACE / "logs" / "submit"
LOG_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("WORKSPACE:", WORKSPACE)
print("Python:", sys.version.split()[0])
print("annotations:", ANNOTATIONS_PATH.exists())
print("GT:", GT_PATH.exists())
print("VIDEO_DIR:", VIDEO_DIR.exists(), list(VIDEO_DIR.glob("*.mp4"))[:3])
print("task_rules:", TASK_RULES_PATH.exists())


WORKSPACE: /workspace
Python: 3.10.12
annotations: True
GT: True
VIDEO_DIR: True [PosixPath('/workspace/data/videos/bkwLxcfcKTY_1_71to281.mp4'), PosixPath('/workspace/data/videos/fabQO-_tS8g_3_0to159.mp4'), PosixPath('/workspace/data/videos/BWFLztKBrLY_3_414to684_scene02.mp4')]
task_rules: True


## Section 2: instruction 読み込み・タスク分解

In [2]:

from src.parse.instruction_parser_ver19 import (
    parse_annotations_jsonl,
    build_predictions,
    build_noun_bank,
    MULTI_CFG_BEST,
    SINGLE_CFG_BEST,
)

# instruction 読み込み
records = parse_annotations_jsonl(ANNOTATIONS_PATH)
print(f"records loaded: {len(records)}")
print("sample:", records[0])


records loaded: 100
sample: {'video_path': 'wyzi9GNZFMU_0_0to121.mp4', 'class': 'Camera Motion Editing', 'subclass': 'Dolly in', 'instruction': "Apply a smooth dolly in effect toward the man's face, starting from the original medium shot and ending in a close-up while keeping him centered.", 'gt_tasks': [], 'gt_primary': {}}


In [3]:

# ver19 GT名詞優先ルーティングでタスク分解（multi mode）
predictions = build_predictions(records, GT_PATH, mode="multi", cfg=MULTI_CFG_BEST)

print(f"predictions: {len(predictions)}")
print("\n--- Sample task decomposition (first 5) ---")
for p in predictions[:5]:
    tasks = p["prediction"]["tasks"]
    print(f"  {p['video_path']}")
    for t in tasks:
        print(f"    [{t['action']}] target={t['target']!r}  params={list(t['params'].keys())}")


predictions: 100

--- Sample task decomposition (first 5) ---
  wyzi9GNZFMU_0_0to121.mp4
    [dolly_in] target="man's face"  params=['motion_type', 'start_framing', 'end_framing']
    [preserve_framing] target="man's face"  params=['position']
  8rKYl1CdXCc_5_276to660_scene02.mp4
    [edit_motion] target='person'  params=['gesture', 'body_part']
    [preserve_identity] target='person'  params=[]
    [preserve_objects] target='his identity'  params=[]
    [stabilize_motion] target='person'  params=[]
    [refine_mask] target='person'  params=[]
  1s9DER1bpm0_10_0to213.mp4
    [add_object] target='rhino and buffalo'  params=['count', 'position', 'spatial_distribution', 'density']
    [match_appearance] target='rhino and buffalo'  params=[]
    [stabilize_instances] target='rhino and buffalo'  params=[]
    [blend_instances] target='rhino and buffalo'  params=[]
  DaUJkmBvTKM_2_0to150.mp4
    [replace_background] target='background behind speaker'  params=['new_scene']
    [preserve_foreg

In [4]:

# subject 混入チェック
subject_count = sum(
    1 for p in predictions
    for t in p["prediction"]["tasks"]
    if "subject" in t.get("target", "")
)
print(f"subject in target count: {subject_count} / {sum(len(p['prediction']['tasks']) for p in predictions)}")
assert subject_count == 0, "subject が target に混入している!"
print("✓ subject ゼロ確認OK")

# タスク分解 JSON をログ保存
task_decomp_path = LOG_DIR / "task_decomposition_ver20_preview.json"
task_decomp_path.write_text(json.dumps(predictions, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"task decomposition saved → {task_decomp_path}")


subject in target count: 0 / 322
✓ subject ゼロ確認OK
task decomposition saved → /workspace/logs/notebooks/task_decomposition_ver20_preview.json


## Section 3: タスクルール確認

In [5]:

from src.submit_baseline_ver05 import load_task_rules, get_action_rule

rules = load_task_rules(TASK_RULES_PATH)
print(f"rules loaded: {len(rules)} actions")

# 各 prediction で使われる action の rule 確認
action_freq = {}
for p in predictions:
    for t in p["prediction"]["tasks"]:
        a = t["action"]
        action_freq[a] = action_freq.get(a, 0) + 1

print("\n--- Action frequency & rules ---")
for action, cnt in sorted(action_freq.items(), key=lambda x: -x[1]):
    rule = get_action_rule(rules, action)
    tool = rule.get("primary_tool", "?")
    method = rule.get("method", "?")
    gpu = "GPU" if rule.get("gpu_required") else "CPU"
    print(f"  {action:<35} x{cnt:3d}  tool={tool:<20} method={method:<25} {gpu}")


rules loaded: 43 actions

--- Action frequency & rules ---
  refine_mask                         x 28  tool=passthrough          method=identity                  CPU
  preserve_objects                    x 18  tool=passthrough          method=identity                  CPU
  stabilize_motion                    x 18  tool=passthrough          method=identity                  CPU
  apply_style                         x 15  tool=opencv               method=stylize                   CPU
  match_lighting                      x 13  tool=opencv               method=histogram_match           CPU
  edit_motion                         x 12  tool=passthrough          method=identity                  CPU
  replace_background                  x 12  tool=sam2_opencv          method=segment_and_replace       GPU
  enhance_style_details               x 12  tool=opencv               method=sharpness                 CPU
  preserve_identity                   x 11  tool=passthrough          method=identity

## Section 4: 動画処理パイプライン動作確認（3本サンプル）

In [6]:

import logging
from src.submit_baseline_ver05 import (
    setup_logger, process_one_video, read_video, write_video,
)

# ノートブック用シンプルロガー
nb_logger = setup_logger(LOG_DIR, "ver20_preview")

# 3本サンプル処理
sample_preds = predictions[:3]
results = []

for row in sample_preds:
    video_name = row["video_path"]
    input_path = VIDEO_DIR / video_name
    if not input_path.exists():
        print(f"SKIP (file not found): {video_name}")
        continue
    try:
        res = process_one_video(row, VIDEO_DIR, OUTPUT_DIR, rules, "mp4v", nb_logger)
        results.append(res)
        print(f"OK  {video_name}  frames={res['input_frames']}→{res['output_frames']}  "
              f"size={res['width']}x{res['height']}  applied={res['applied_actions']}")
    except Exception as e:
        print(f"ERR {video_name}: {e}")

print(f"\n処理完了: {len(results)} / {len(sample_preds)}")


2026-03-29 12:52:46,572 [INFO] [START] wyzi9GNZFMU_0_0to121.mp4  tasks=['dolly_in', 'preserve_framing']
2026-03-29 12:52:48,209 [INFO] [DONE] wyzi9GNZFMU_0_0to121.mp4  applied=['dolly_in']  skipped=[]
OK  wyzi9GNZFMU_0_0to121.mp4  frames=120→120  size=1920x1080  applied=['dolly_in']
2026-03-29 12:52:48,216 [INFO] [START] 8rKYl1CdXCc_5_276to660_scene02.mp4  tasks=['edit_motion', 'preserve_identity', 'preserve_objects', 'stabilize_motion', 'refine_mask']
2026-03-29 12:52:49,104 [INFO] [DONE] 8rKYl1CdXCc_5_276to660_scene02.mp4  applied=[]  skipped=[]
OK  8rKYl1CdXCc_5_276to660_scene02.mp4  frames=125→125  size=1920x1080  applied=[]
2026-03-29 12:52:49,112 [INFO] [START] 1s9DER1bpm0_10_0to213.mp4  tasks=['add_object', 'match_appearance', 'stabilize_instances', 'blend_instances']
2026-03-29 12:52:51,010 [INFO] [DONE] 1s9DER1bpm0_10_0to213.mp4  applied=['match_appearance']  skipped=[]
OK  1s9DER1bpm0_10_0to213.mp4  frames=120→120  size=1920x1080  applied=['match_appearance']

処理完了: 3 / 3


## Section 5: frame数・解像度の一致確認

In [7]:

import cv2 as cv2_check

print("--- Frame数 / 解像度検証 ---")
all_pass = True
for res in results:
    video_name = res["video_path"]
    output_path = OUTPUT_DIR / video_name
    input_path = VIDEO_DIR / video_name

    # 入力
    cap_in = cv2_check.VideoCapture(str(input_path))
    in_frames = int(cap_in.get(cv2_check.CAP_PROP_FRAME_COUNT))
    in_w = int(cap_in.get(cv2_check.CAP_PROP_FRAME_WIDTH))
    in_h = int(cap_in.get(cv2_check.CAP_PROP_FRAME_HEIGHT))
    cap_in.release()

    # 出力
    cap_out = cv2_check.VideoCapture(str(output_path))
    out_frames = int(cap_out.get(cv2_check.CAP_PROP_FRAME_COUNT))
    out_w = int(cap_out.get(cv2_check.CAP_PROP_FRAME_WIDTH))
    out_h = int(cap_out.get(cv2_check.CAP_PROP_FRAME_HEIGHT))
    cap_out.release()

    frames_ok = in_frames == out_frames
    size_ok = (in_w == out_w) and (in_h == out_h)
    status = "✓" if (frames_ok and size_ok) else "✗"
    if not (frames_ok and size_ok):
        all_pass = False
    print(f"  {status} {video_name}")
    print(f"     frames: {in_frames} → {out_frames}  {'OK' if frames_ok else 'NG'}")
    print(f"     size  : {in_w}x{in_h} → {out_w}x{out_h}  {'OK' if size_ok else 'NG'}")

print(f"\nAll pass: {all_pass}")
assert all_pass, "frame数または解像度が一致していない!"


--- Frame数 / 解像度検証 ---
  ✓ wyzi9GNZFMU_0_0to121.mp4
     frames: 120 → 120  OK
     size  : 1920x1080 → 1920x1080  OK
  ✓ 8rKYl1CdXCc_5_276to660_scene02.mp4
     frames: 125 → 125  OK
     size  : 1920x1080 → 1920x1080  OK
  ✓ 1s9DER1bpm0_10_0to213.mp4
     frames: 120 → 120  OK
     size  : 1920x1080 → 1920x1080  OK

All pass: True


## Section 6: 全100本パイプライン実行 + 結果まとめ

In [ ]:

import traceback as tb_module

# 全100本処理
all_results = []
all_errors = []

OUTPUT_DIR_FULL = WORKSPACE / "logs" / "submit" / "submission_ver05_videos"
OUTPUT_DIR_FULL.mkdir(parents=True, exist_ok=True)

print(f"Processing {len(predictions)} videos...")
for i, row in enumerate(predictions):
    video_name = row["video_path"]
    input_path = VIDEO_DIR / video_name
    if not input_path.exists():
        print(f"[{i+1:3d}] MISSING: {video_name}")
        all_errors.append({"video_path": video_name, "error": "input not found"})
        continue
    try:
        res = process_one_video(row, VIDEO_DIR, OUTPUT_DIR_FULL, rules, "mp4v", nb_logger)
        all_results.append(res)
        print(f"[{i+1:3d}] OK  {video_name}  applied={res['applied_actions']}  skipped={res['skipped_actions']}")
    except Exception as e:
        print(f"[{i+1:3d}] ERR {video_name}: {e}")
        all_errors.append({"video_path": video_name, "error": str(e)})

print(f"\n=== 完了 ===")
print(f"  成功: {len(all_results)} / {len(predictions)}")
print(f"  失敗: {len(all_errors)}")


Processing 100 videos...
2026-03-29 12:53:00,737 [INFO] [START] wyzi9GNZFMU_0_0to121.mp4  tasks=['dolly_in', 'preserve_framing']
2026-03-29 12:53:02,244 [INFO] [DONE] wyzi9GNZFMU_0_0to121.mp4  applied=['dolly_in']  skipped=[]
[  1] OK  wyzi9GNZFMU_0_0to121.mp4  applied=['dolly_in']  skipped=[]
2026-03-29 12:53:02,258 [INFO] [START] 8rKYl1CdXCc_5_276to660_scene02.mp4  tasks=['edit_motion', 'preserve_identity', 'preserve_objects', 'stabilize_motion', 'refine_mask']
2026-03-29 12:53:03,165 [INFO] [DONE] 8rKYl1CdXCc_5_276to660_scene02.mp4  applied=[]  skipped=[]
[  2] OK  8rKYl1CdXCc_5_276to660_scene02.mp4  applied=[]  skipped=[]
2026-03-29 12:53:03,173 [INFO] [START] 1s9DER1bpm0_10_0to213.mp4  tasks=['add_object', 'match_appearance', 'stabilize_instances', 'blend_instances']
2026-03-29 12:53:05,050 [INFO] [DONE] 1s9DER1bpm0_10_0to213.mp4  applied=['match_appearance']  skipped=[]
[  3] OK  1s9DER1bpm0_10_0to213.mp4  applied=['match_appearance']  skipped=[]
2026-03-29 12:53:05,064 [INFO] [S

In [ ]:

# 結果集計・ログ保存
import collections

action_stats = collections.Counter()
skipped_actions_total = collections.Counter()
for r in all_results:
    for a in r.get("applied_actions", []):
        action_stats[a.replace("[fallback]", "")] += 1
    for a in r.get("skipped_actions", []):
        skipped_actions_total[a] += 1

print("--- Applied action stats ---")
for a, cnt in action_stats.most_common():
    print(f"  {a:<40} {cnt}")

print("\n--- Skipped action stats ---")
for a, cnt in skipped_actions_total.most_common():
    print(f"  {a:<40} {cnt}")

# manifest 保存
manifest_out = {
    "total": len(predictions),
    "success": len(all_results),
    "errors": len(all_errors),
    "applied_action_stats": dict(action_stats),
    "skipped_action_stats": dict(skipped_actions_total),
    "error_list": all_errors,
    "results": all_results,
}
manifest_path = LOG_DIR / "manifest_ver20.json"
manifest_path.write_text(json.dumps(manifest_out, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"\nmanifest saved → {manifest_path}")


## Section 7: 出力 ZIP 作成 + 最終検証

In [ ]:

import zipfile

# ZIP 作成
OUTPUT_ZIP = WORKSPACE / "logs" / "submit" / "submission_ver05.zip"
with zipfile.ZipFile(OUTPUT_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for mp4 in sorted(OUTPUT_DIR_FULL.glob("*.mp4")):
        zf.write(mp4, arcname=mp4.name)

zip_size_mb = OUTPUT_ZIP.stat().st_size / (1024 * 1024)
actual_mp4s = list(OUTPUT_DIR_FULL.glob("*.mp4"))
print(f"ZIP: {OUTPUT_ZIP}  ({zip_size_mb:.1f} MB)")
print(f"MP4 count in zip: {len(actual_mp4s)}")

# 最終検証
expected_videos = {r["video_path"] for r in predictions}
actual_videos = {p.name for p in actual_mp4s}
missing = sorted(expected_videos - actual_videos)
extra = sorted(actual_videos - expected_videos)
print(f"\n--- Final validation ---")
print(f"  expected: {len(expected_videos)}")
print(f"  actual  : {len(actual_videos)}")
print(f"  missing : {missing}")
print(f"  extra   : {extra}")
print(f"  status  : {'OK' if not missing else 'INCOMPLETE'}")

# まとめレポート保存
final_report = {
    "pipeline": "submit_baseline_ver05",
    "instruction_parser": "instruction_parser_ver19",
    "task_rules": "task_rules_ver05.json",
    "records": len(predictions),
    "success": len(all_results),
    "errors": len(all_errors),
    "missing_in_output": missing,
    "zip": str(OUTPUT_ZIP),
    "zip_size_mb": round(zip_size_mb, 2),
    "subject_in_target": 0,
    "action_stats": dict(action_stats),
}
report_path = LOG_DIR / "final_report_ver20.json"
report_path.write_text(json.dumps(final_report, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"\nfinal report saved → {report_path}")
